# AnyCount — Apple Blend Pack (COCO + dense), Colab

Notebook lanjutan dari `apple_coco_colab.ipynb`. Tujuannya satu pack yang kuat di dua
domain: apel "biasa" (COCO) **dan** tumpukan padat (pseudo-label + komposit sintetis).

Alur: COCO subset → + `apples.jpg` padat (289 titik LLM dari repo) → + komposit sintetis
(40–80 apel per kanvas, label eksak) → latih YOLO11s → evaluasi sparse + dense →
bandingkan dengan pack `apple-coco` v1 yang sekarang → ekspor ONNX INT8.

**Cara pakai:** `Runtime → Change runtime type → T4 GPU` → `Run all`. ±50–80 menit.

Catatan: `apples.jpg` ikut training, jadi angka di situ bukan uji; uji dense memakai
komposit sintetis yang di-holdout dengan jumlah objek eksak.

## 1) Dependensi

In [ ]:
!pip -q install "ultralytics>=8.3" onnx onnxruntime pillow requests tqdm

import sys, torch, ultralytics
print("python:", sys.version.split()[0])
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("PERINGATAN: GPU tidak aktif. Runtime > Change runtime type > T4 GPU, lalu Run all lagi.")

## 2) Konfigurasi

In [ ]:
from pathlib import Path
import random

CLASS_NAME = "apple"
MODEL_WEIGHTS = "yolo11s.pt"
EPOCHS = 70
IMG_SIZE = 640
BATCH = 16
COCO_TRAIN = 1200
COCO_VAL = 250
SYNTH_TRAIN = 180       # jumlah kanvas komposit dense untuk training
SYNTH_TEST = 20         # kanvas holdout untuk uji count eksak
SYNTH_MIN_OBJ = 40
SYNTH_MAX_OBJ = 80
MIN_BOX_PX = 4
SEED = 7
PACK_SLUG = "apple-blend"

REPO_RAW = "https://raw.githubusercontent.com/Synthever/anycount/main"
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/anycount_packs"

WORK = Path("/content/anycount-blend")
COCO = WORK / "coco"
DS = WORK / "dataset"
RUNS = WORK / "runs"
WORK.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
print("workdir:", WORK)

## 3) Unduh anotasi COCO 2017

In [ ]:
import shutil
import urllib.request, zipfile

ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANN_ZIP = COCO / "annotations_trainval2017.zip"
ANN_DIR = COCO / "annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)

if not ANN_ZIP.exists():
    print("mengunduh anotasi (~241 MB)...")
    urllib.request.urlretrieve(ANN_URL, ANN_ZIP)

needed = {"instances_train2017.json", "instances_val2017.json"}
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in z.namelist():
        base = Path(name).name
        target = ANN_DIR / base
        if base in needed and not target.exists():
            print("extract", base)
            with z.open(name) as fsrc, target.open("wb") as fdst:
                shutil.copyfileobj(fsrc, fdst, length=1 << 20)

found = sorted(p.name for p in ANN_DIR.glob("*.json"))
print(found)
if not needed.issubset(set(found)):
    raise SystemExit(f"anotasi tidak lengkap: {found}")

## 4) COCO subset (hanya gambar berisi apple)

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm.auto import tqdm

def load_split(split):
    with (ANN_DIR / f"instances_{split}2017.json").open() as f:
        return json.load(f)

def target_category(data):
    matches = [c for c in data["categories"] if c["name"] == CLASS_NAME]
    if not matches:
        raise SystemExit(f"kelas {CLASS_NAME!r} tidak ada di COCO")
    return matches[0]["id"]

def collect(split, limit):
    data = load_split(split)
    cat_id = target_category(data)
    images = {im["id"]: im for im in data["images"]}
    boxes = {}
    for ann in data["annotations"]:
        if ann["category_id"] != cat_id or ann.get("iscrowd"):
            continue
        x, y, w, h = ann["bbox"]
        if w < MIN_BOX_PX or h < MIN_BOX_PX:
            continue
        boxes.setdefault(ann["image_id"], []).append((x, y, w, h))
    ids = sorted(boxes)
    if limit and len(ids) > limit:
        ids = random.sample(ids, limit)
    return [(images[i], boxes[i]) for i in ids]

train_items = collect("train", COCO_TRAIN)
val_items = collect("val", COCO_VAL)
print(f"kandidat train: {len(train_items)} | val: {len(val_items)}")

def download(item, split):
    im, _ = item
    dst = DS / "images" / split / im["file_name"]
    if dst.exists():
        return dst
    url = f"http://images.cocodataset.org/{split}2017/{im['file_name']}"
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            dst.write_bytes(r.content)
            return dst
        except Exception:
            if attempt == 2:
                raise
    return dst

def materialize(items, split):
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    written = 0
    with ThreadPoolExecutor(max_workers=16) as pool:
        futs = {pool.submit(download, it, split): it for it in items}
        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"download {split}"):
            im, boxes = futs[fut]
            fut.result()
            w, h = im["width"], im["height"]
            lines = [
                f"0 {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}"
                for x, y, bw, bh in boxes
            ]
            (DS / "labels" / split / f"{Path(im['file_name']).stem}.txt").write_text(
                "\n".join(lines) + "\n"
            )
            written += 1
    return written

n_train = materialize(train_items, "train")
n_val = materialize(val_items, "val")
print(f"coco train: {n_train} | coco val: {n_val}")

## 5) Domain padat: `apples.jpg` + pseudo-label LLM (289 titik) dari repo

In [ ]:
import io
import numpy as np
from PIL import Image

IMG_URL = f"{REPO_RAW}/research/samples/apples.jpg"
LBL_URL = f"{REPO_RAW}/research/spikes/004-offline-onnx-llm-validator/results/apples__llm_validator.json"

dense_img = Image.open(io.BytesIO(requests.get(IMG_URL, timeout=60).content)).convert("RGB")
dense_rec = requests.get(LBL_URL, timeout=60).json()

def pseudo_boxes(points, w, h):
    pts = np.array([(float(p["x"]), float(p["y"])) for p in points], dtype=float)
    if len(pts) == 0:
        return []
    min_px, max_px = 10.0, 0.30 * min(w, h)
    out = []
    for i in range(len(pts)):
        d = np.hypot(pts[:, 0] - pts[i, 0], pts[:, 1] - pts[i, 1])
        d[i] = np.inf
        k = min(3, max(1, len(pts) - 1))
        d_nn = float(np.median(np.sort(d)[:k])) if len(pts) > 1 else min_px / 0.92
        size = float(np.clip(d_nn * 0.92, min_px, max_px))
        out.append((float(pts[i, 0]), float(pts[i, 1]), size))
    return out

def write_dense(stem, img, boxes):
    img.save(DS / "images" / "train" / f"{stem}.jpg", quality=92)
    w, h = img.size
    lines = [
        f"0 {cx / w:.6f} {cy / h:.6f} {size / w:.6f} {size / h:.6f}"
        for cx, cy, size in boxes
    ]
    (DS / "labels" / "train" / f"{stem}.txt").write_text("\n".join(lines) + "\n")

w, h = dense_img.size
dense_boxes = pseudo_boxes(dense_rec["points"], w, h)
write_dense("dense_apples_full", dense_img, dense_boxes)
print("dense apples full:", len(dense_boxes), "box")

# 4 tile 2x2 dengan overlap 8%, box direproyeksikan
ow, oh = round(w * 0.08), round(h * 0.08)
for r in range(2):
    for c in range(2):
        x0 = max(0, round(c * w / 2) - ow)
        x1 = min(w, round((c + 1) * w / 2) + ow)
        y0 = max(0, round(r * h / 2) - oh)
        y1 = min(h, round((r + 1) * h / 2) + oh)
        tw, th = x1 - x0, y1 - y0
        kept = [
            (cx - x0, cy - y0, size)
            for cx, cy, size in dense_boxes
            if x0 <= cx < x1 and y0 <= cy < y1
        ]
        if kept:
            write_dense(f"dense_apples_t{r}{c}", dense_img.crop((x0, y0, x1, y1)), kept)
            print(f"tile r{r}c{c}:", len(kept), "box")

## 6) Komposit sintetis padat (label eksak, holdout terpisah)

In [ ]:
# Kumpulkan potongan apel dari gambar COCO + labelnya.
crops = []
for img_path in sorted((DS / "images" / "train").glob("*.jpg")):
    if img_path.stem.startswith("dense_"):
        continue
    lbl = DS / "labels" / "train" / f"{img_path.stem}.txt"
    if not lbl.exists():
        continue
    im = Image.open(img_path).convert("RGB")
    iw, ih = im.size
    for line in lbl.read_text().splitlines():
        cls, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx) * iw, float(cy) * ih, float(bw) * iw, float(bh) * ih
        half = int(max(max(bw, bh) * 0.62, 12))
        x0, y0 = int(max(0, cx - half)), int(max(0, cy - half))
        x1, y1 = int(min(iw, cx + half)), int(min(ih, cy + half))
        if x1 - x0 >= 20 and y1 - y0 >= 20:
            crops.append(im.crop((x0, y0, x1, y1)))
print("potongan apel:", len(crops))
if not crops:
    raise SystemExit("tidak ada crop apel; cek dataset COCO")

backgrounds = [
    Image.open(p).convert("RGB")
    for p in sorted((DS / "images" / "train").glob("*.jpg"))
    if not p.stem.startswith("dense_")
]

def make_composite(index, subset, prefix, n_obj):
    bg = random.choice(backgrounds).copy()
    if min(bg.size) < 600:
        k = 600 / min(bg.size)
        bg = bg.resize((round(bg.size[0] * k), round(bg.size[1] * k)), Image.LANCZOS)
    bw, bh = bg.size
    lines = []
    for _ in range(n_obj):
        crop = random.choice(crops)
        size = random.randint(26, 64)
        patch = crop.resize((size, size), Image.LANCZOS)
        cx = random.uniform(size, bw - size)
        cy = random.uniform(size, bh - size)
        bg.paste(patch, (int(cx - size / 2), int(cy - size / 2)))
        lines.append(f"0 {cx / bw:.6f} {cy / bh:.6f} {size / bw:.6f} {size / bh:.6f}")
    if subset == "train":
        img_dir, lbl_dir = DS / "images" / "train", DS / "labels" / "train"
    else:
        img_dir, lbl_dir = DS / "holdout" / "images", DS / "holdout" / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    bg.save(img_dir / f"{prefix}{index:04d}.jpg", quality=90)
    (lbl_dir / f"{prefix}{index:04d}.txt").write_text("\n".join(lines) + "\n")
    return n_obj

for sub in ("images/train", "labels/train", "holdout/images", "holdout/labels"):
    (DS / sub).mkdir(parents=True, exist_ok=True)

synth_counts = {}
for i in range(SYNTH_TRAIN):
    n = random.randint(SYNTH_MIN_OBJ, SYNTH_MAX_OBJ)
    synth_counts[f"synth_{i:04d}"] = make_composite(i, "train", "synth_", n)

random.seed(SEED + 1)
test_counts = {}
for i in range(SYNTH_TEST):
    n = random.randint(SYNTH_MIN_OBJ, SYNTH_MAX_OBJ)
    test_counts[f"synth_test_{i:04d}"] = make_composite(i, "holdout", "synth_test_", n)
(DS / "synth_test_counts.json").write_text(json.dumps(test_counts, indent=2))
print("komposit train:", len(synth_counts), "| holdout:", len(test_counts))

## 7) `data.yaml` + statistik

In [ ]:
yaml_path = DS / "data.yaml"
yaml_path.write_text(
    f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: {CLASS_NAME}\n"
)
print(yaml_path.read_text())

tr = len(list((DS / "images" / "train").glob("*.jpg")))
va = len(list((DS / "images" / "val").glob("*.jpg")))
print("train images:", tr, "| val images:", va)
print("holdout synth:", len(test_counts), "kanvas dengan count eksak (tidak masuk data.yaml)")

## 8) Training (blend sparse + dense)

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL_WEIGHTS)
model.train(
    data=str(DS / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=device,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=0.3,
    close_mosaic=10,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    fliplr=0.5,
    patience=20,
    workers=2,
    project=str(RUNS),
    name=f"{PACK_SLUG}_v1",
    exist_ok=True,
    seed=SEED,
    verbose=True,
)
best = RUNS / f"{PACK_SLUG}_v1" / "weights" / "best.pt"
print("best:", best, "| ada:", best.exists())

## 9) Evaluasi: mAP COCO + count pada holdout dense vs pack `apple-coco` v1

In [ ]:
metrics = model.val(data=str(DS / "data.yaml"), imgsz=IMG_SIZE, device=device)
print(f"mAP50={metrics.box.map50:.3f} mAP50-95={metrics.box.map:.3f}")

import numpy as np
import onnxruntime as ort

def letterbox(im, imgsz=640):
    w, h = im.size
    s = min(imgsz / w, imgsz / h)
    nw, nh = round(w * s), round(h * s)
    canvas = Image.new("RGB", (imgsz, imgsz), (114, 114, 114))
    canvas.paste(im.resize((nw, nh), Image.BILINEAR), ((imgsz - nw) // 2, (imgsz - nh) // 2))
    arr = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    return arr, s, (imgsz - nw) // 2, (imgsz - nh) // 2

def box_nms(boxes, scores, thr=0.45):
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = int(order[0])
        keep.append(i)
        if order.size == 1:
            break
        rest = order[1:]
        xx1 = np.maximum(boxes[i, 0], boxes[rest, 0])
        yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])
        xx2 = np.minimum(boxes[i, 2], boxes[rest, 2])
        yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])
        inter = np.maximum(0.0, xx2 - xx1) * np.maximum(0.0, yy2 - yy1)
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_r = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])
        iou = inter / (area_i + area_r - inter + 1e-6)
        order = rest[iou <= thr]
    return keep

def count_onnx(sess, inp_name, im, imgsz=640, conf=0.25):
    blob, s, px, py = letterbox(im, imgsz)
    out = sess.run(None, {inp_name: blob})[0]
    preds = np.squeeze(out, axis=0)
    if preds.shape[0] < preds.shape[1]:
        preds = preds.T
    scores = preds[:, 4]
    mask = scores >= conf
    preds, scores = preds[mask], scores[mask]
    if len(preds) == 0:
        return 0
    cx, cy, bw, bh = preds[:, 0], preds[:, 1], preds[:, 2], preds[:, 3]
    boxes = np.stack([
        (cx - bw / 2 - px) / s, (cy - bh / 2 - py) / s,
        (cx + bw / 2 - px) / s, (cy + bh / 2 - py) / s,
    ], axis=1)
    return len(box_nms(boxes, scores, 0.45))

# baseline ONNX dari repo
baseline_url = f"{REPO_RAW}/research/packs/apple-coco/models/apple_coco_v1_int8.onnx"
baseline_path = WORK / "apple_coco_v1_int8.onnx"
if not baseline_path.exists():
    baseline_path.write_bytes(requests.get(baseline_url, timeout=120).content)
base_sess = ort.InferenceSession(str(baseline_path), providers=["CPUExecutionProvider"])
base_inp = base_sess.get_inputs()[0].name

def dense_eval():
    rows = []
    for stem, truth in sorted(test_counts.items()):
        p = DS / "holdout" / "images" / f"{stem}.jpg"
        im = Image.open(p).convert("RGB")
        res = model.predict(str(p), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
        ft = 0 if res.boxes is None else len(res.boxes)
        base = count_onnx(base_sess, base_inp, im)
        rows.append((stem, truth, ft, base))
    return rows

rows = dense_eval()
mae_ft = float(np.mean([abs(r[2] - r[1]) for r in rows]))
mae_base = float(np.mean([abs(r[3] - r[1]) for r in rows]))
mape_ft = 100 * float(np.mean([abs(r[2] - r[1]) / r[1] for r in rows]))
mape_base = 100 * float(np.mean([abs(r[3] - r[1]) / r[1] for r in rows]))
print(f"holdout dense ({len(rows)} kanvas): fine-tuned MAE={mae_ft:.1f} ({mape_ft:.1f}%) "
      f"| apple-coco v1 MAE={mae_base:.1f} ({mape_base:.1f}%)")
for r in rows[:5]:
    print("  ", r)

## 10) Ekspor ONNX INT8 (+ best.pt untuk fine-tune berikutnya)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

model = YOLO(str(best))
onnx_fp32 = Path(model.export(format="onnx", imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True))
int8_path = WORK / f"{PACK_SLUG}_v1_int8.onnx"
quantize_dynamic(str(onnx_fp32), str(int8_path), weight_type=QuantType.QInt8)
print(f"FP32: {onnx_fp32.stat().st_size / 1e6:.2f} MB | INT8: {int8_path.stat().st_size / 1e6:.2f} MB")

## 11) Kemas + unduh

In [ ]:
import json, shutil, zipfile

summary = {
    "class": CLASS_NAME,
    "model": MODEL_WEIGHTS,
    "epochs": EPOCHS,
    "imgsz": IMG_SIZE,
    "coco_train": n_train,
    "coco_val": n_val,
    "synth_train": len(synth_counts),
    "synth_test": len(test_counts),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "dense_mae_finetuned": round(mae_ft, 2),
    "dense_mape_finetuned_pct": round(mape_ft, 2),
    "dense_mae_apple_coco_v1": round(mae_base, 2),
    "dense_mape_apple_coco_v1_pct": round(mape_base, 2),
    "onnx_int8_mb": round(int8_path.stat().st_size / 1e6, 2),
}
(WORK / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

artifact = WORK / f"{PACK_SLUG}_v1_bundle.zip"
run_dir = RUNS / f"{PACK_SLUG}_v1"
with zipfile.ZipFile(artifact, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(int8_path, int8_path.name)
    z.write(WORK / "summary.json", "summary.json")
    z.write(run_dir / "results.csv", "results.csv")
    z.write(run_dir / "weights" / "best.pt", "best.pt")
    for p in sorted(run_dir.glob("*.png")):
        z.write(p, f"plots/{p.name}")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy(int8_path, Path(DRIVE_DIR) / int8_path.name)
    shutil.copy(artifact, Path(DRIVE_DIR) / artifact.name)

try:
    from google.colab import files
    files.download(str(int8_path))
    files.download(str(artifact))
except Exception as exc:
    print("auto-download tidak tersedia:", exc)

print("Selesai. Kirim .onnx + bundle (berisi best.pt) ke agent.")